# Active vision with deep neural networks: hands-on

In passive vision models, the network sees the whole image at once. A model with active vision instead sees it via a sequence of **glimpses**, and has to decide **where to look next**. In this notebook we develop active vision agents.

Sections:

- **A. A policy is representative of how the eye moves across the scene.** How glimpses are sampled, and what inhibition of return does.
- **B. Learn to represent the scene via next step observation prediction ([seq-JEPA](https://openreview.net/forum?id=MO1OLAKcsJ)).** A model that learns from glimpses in a self-supervised manner by predicting what it will see after each eye
  movement. You will train it for a few steps, then load the fully trained model and test it.
- **C. What does seq-JEPA encode?** Linear probes on the model's representations: does the model know *where* it is,
  and *what* is in the image?
- **D. What does it cost?** Comparing the computational cost of processing Glimpses against the full images.
- **E. Learning where to look based on task goals.** A policy that uses the frozen model to search for a cued object,
  trained with reinforcement learning.

Every section consists of some **tasks**: *coding exercises*, where you replace the `...` in a function and remove its `raise NotImplementedError` line, and some *Think!* questions. The point of these tasks is to understand how the system works. Each task is followed by a collapsed **Solution** (click it to open), and each coding exercise also by a cell that loads our solution, so the rest of the notebook runs even if you are stuck. Please try to solve each task yourself, first.

Everything runs on a laptop CPU; and the whole notebook takes at most few minutes to run.

You need to have Python 3.9 or newer with PyTorch 2.1 or newer (the CPU build is enough), and the following libraries: torchvision, numpy, matplotlib
and pillow

**Data.** Download the data pack `active_vision_data.zip` (about 380 MB) from
[Google Drive](https://drive.google.com/file/d/1MDNFCfCaYJypfQL5N4S_axrlLqXkyUWB/view?usp=sharing) and unzip it next to this notebook, so that the
folder `active_vision_data/` sits beside it.

In [ ]:
DATA_DIR = 'active_vision_data'    # the data pack, unzipped next to this notebook
SEED = 0

In [ ]:
import ast
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from matplotlib.patches import Rectangle
from PIL import Image

torch.manual_seed(SEED)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10, 'axes.titlesize': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.edgecolor': '#9a9893', 'axes.labelcolor': '#52514e',
                     'xtick.color': '#52514e', 'ytick.color': '#52514e', 'legend.frameon': False})
INK, INK2, INK3, GRIDC = '#0b0b0b', '#52514e', '#9a9893', '#e4e3df'
POLICY_COLORS = {'uniform': '#3b5bdb', 'saliency': '#0c9fb5', 'saliency + IoR': '#b5179e'}

if not os.path.exists(os.path.join(DATA_DIR, 'meta.json')) and os.path.exists(os.path.join(DATA_DIR, 'active_vision_data')):
    DATA_DIR = os.path.join(DATA_DIR, 'active_vision_data')   # unzipped into a folder of the same name (Windows "Extract All")
if not os.path.exists(os.path.join(DATA_DIR, 'meta.json')):
    raise FileNotFoundError(f'Data pack not found in {os.path.abspath(DATA_DIR)}. Unzip active_vision_data.zip there.')
META = json.load(open(os.path.join(DATA_DIR, 'meta.json')))
IMAGES, TRIALS = META['images'], META['trials']


def load_solution(name):
    """Define our solution of the exercise function `name` (from the data pack), e.g. if you are stuck."""
    src = open(os.path.join(DATA_DIR, 'solutions.py')).read()
    fn = next(n for n in ast.parse(src).body if getattr(n, 'name', None) == name)
    start = (fn.decorator_list[0] if fn.decorator_list else fn).lineno
    exec('\n'.join(src.splitlines()[start - 1:fn.end_lineno]), globals())


SALIENCY = np.load(os.path.join(DATA_DIR, 'saliency_g128.npy'))
FRAME, GRID, FOVEA = META['frame'], META['grid'], META['fovea']        # 512-px frame, 10 x 10 cells, 128-px fovea
print(f'torch {torch.__version__} on CPU ({torch.get_num_threads()} threads) | {len(IMAGES)} images, '
      f'{len(TRIALS)} search trials')

## A. A policy is representative of how the eye moves across the scene

The images shown in this section are from [MS-COCO](https://cocodataset.org/#home) dataset. Each full image is 512 x 512 frame. A **glimpse** is a
128 x 128 crop around the current fixation, i.e., one sixteenth of the frame.

For pre-training our models with glimpse, we use one of the three rules below:

- **uniform**: all cells equally likely to be sampled for each glimpse.
- **saliency**: in proportion to the [DeepGaze IIE](https://ieeexplore.ieee.org/document/9711473) saliency map, which predicts where people look in free viewing;
- **saliency + IoR**: in proportion to the DeepGaze IIE saliency map, with *inhibition of return*: ie once a place has been looked at, it cannot be revisited in the future.

Here is one image and its map.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
SAL_GRID = SALIENCY.shape[-1]
CELL = FRAME // SAL_GRID                                                # 4 px per saliency cell


def load_frame(rec):
    """COCO image -> uint8 [3, 512, 512] letterboxed frame, exactly as in training."""
    img = Image.open(os.path.join(DATA_DIR, rec['file'])).convert('RGB')
    w, h = img.size
    s = FRAME / max(w, h)
    nw, nh = int(round(w * s)), int(round(h * s))
    img = TF.resize(img, (nh, nw), antialias=True)
    pl, pt = (FRAME - nw) // 2, (FRAME - nh) // 2
    return TF.pil_to_tensor(TF.pad(img, (pl, pt, FRAME - nw - pl, FRAME - nh - pt), fill=0))


def glimpse(frame, row, col):
    """The 128-px crop centred on (row, col), zero-padded where it leaves the frame."""
    h = FOVEA // 2
    out = torch.zeros(3, FOVEA, FOVEA, dtype=frame.dtype)
    r0, c0 = row - h, col - h
    rs, cs, re, ce = max(0, r0), max(0, c0), min(FRAME, r0 + FOVEA), min(FRAME, c0 + FOVEA)
    out[:, rs - r0:re - r0, cs - c0:ce - c0] = frame[:, rs:re, cs:ce]
    return out


def glimpses(frame, fix):
    return torch.stack([glimpse(frame, int(r), int(c)) for r, c in fix])


def normalize(x):
    """uint8 glimpses -> the encoder's input (ImageNet normalization)."""
    return (x.float() / 255 - MEAN) / STD


def content_mask(rec):
    """Saliency cells that overlap the image rather than the black bars."""
    s = FRAME / max(rec['width'], rec['height'])
    nw, nh = int(round(rec['width'] * s)), int(round(rec['height'] * s))
    pl, pt = (FRAME - nw) // 2, (FRAME - nh) // 2
    k = np.arange(SAL_GRID)
    return (((k + 1) * CELL > pt) & (k * CELL < pt + nh))[:, None] & (((k + 1) * CELL > pl) & (k * CELL < pl + nw))[None]


def saliency_overlay(rec):
    """RGBA image of the saliency map at frame resolution: transparent where saliency is low."""
    s = torch.from_numpy(SALIENCY[rec['sal_row']].astype(np.float32))[None, None]
    s = F.interpolate(s, size=(FRAME, FRAME), mode='bilinear', align_corners=False)[0, 0].numpy()
    s = np.clip(s / np.percentile(s, 99.5), 0, 1)
    rgba = plt.get_cmap('magma')(s)
    rgba[..., 3] = 0.8 * s ** 0.7
    return rgba


def show_image(ax, rec, saliency=False, title=''):
    ax.imshow(load_frame(rec).permute(1, 2, 0))
    if saliency:
        ax.imshow(saliency_overlay(rec))
    ax.set_xlim(0, FRAME); ax.set_ylim(FRAME, 0); ax.axis('off'); ax.set_title(title)


IMAGE = 7                                  # try other images: any number below len(IMAGES)
rec = IMAGES[IMAGE]
fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
show_image(ax[0], rec, title='the original image')
show_image(ax[1], rec, saliency=True, title='DeepGaze IIE saliency layered over the image')
show_image(ax[2], rec, title='one glimpse = 128 x 128 px')
ax[2].add_patch(Rectangle((256 - 64, 256 - 64), 128, 128, fill=False, ec='#eda100', lw=2))
plt.tight_layout(); plt.show()

**The sampler.** `sample_fixations` below picks the fixations of one scanpath, one after the other:

- `policy='saliency'`: a cell with probability proportional to its saliency;
- `policy='uniform'`: any cell of the image, all with the same probability.

The fixation is then put on a random pixel inside that cell.

With inhibition of return (`ior=True`), after each fixation the saliency cells under the 128 x 128 window around
it are multiplied by a tiny number (`SUPPRESS`), so the sampler (almost) never picks them again. We do not set
them exactly to zero, so that the sampler still has something to pick once the whole image has been looked at.

In [ ]:
FLOOR, SUPPRESS = 1e-6, 1e-12


def sample_fixations(rec, n, policy='saliency', ior=True, gen=None):
    """n fixations (row, col) in frame pixels, from 'uniform' or 'saliency', with or without IoR."""
    gen = gen or np.random.default_rng()
    inside = content_mask(rec)                                # saliency cells on the image (not on the black bars)
    base = SALIENCY[rec['sal_row']].astype(np.float64) if policy == 'saliency' else inside.astype(np.float64)
    base = base + FLOOR * base[inside].mean() * inside        # a tiny floor: every image cell stays possible
    p, half = base.copy(), FOVEA // 2
    out = []
    for _ in range(n):
        idx = gen.choice(SAL_GRID * SAL_GRID, p=(p / p.sum()).ravel())       # pick a cell
        r, c = (idx // SAL_GRID) * CELL + gen.integers(CELL), (idx % SAL_GRID) * CELL + gen.integers(CELL)
        out.append((int(r), int(c)))
        if ior:                                               # switch off the cells under the 128 x 128 window
            r0, r1 = max(0, (r - half) // CELL), min(SAL_GRID - 1, (r + half) // CELL)
            c0, c1 = max(0, (c - half) // CELL), min(SAL_GRID - 1, (c + half) // CELL)
            p[r0:r1 + 1, c0:c1 + 1] = base[r0:r1 + 1, c0:c1 + 1] * SUPPRESS
    return np.array(out)

Now the three rules side by side, eight glimpses each. With IoR (right), the fixations spread over the
image instead of piling up on the brightest spot. Below the panels: the eight glimpses of the IoR scanpath,
which is all the model ever gets to see of this image.

In [ ]:
def draw_scanpath(ax, rec, fix, color, title='', saliency=True):
    show_image(ax, rec, saliency=saliency, title=title)
    for r, c in fix:
        ax.add_patch(Rectangle((c - FOVEA / 2, r - FOVEA / 2), FOVEA, FOVEA, fill=False, ec=color, lw=1, alpha=0.7))
    ax.plot(fix[:, 1], fix[:, 0], '-', color='w', lw=1.4, alpha=0.9)
    for i, (r, c) in enumerate(fix):
        ax.text(c, r, str(i + 1), color='w', fontsize=8, ha='center', va='center', fontweight='bold',
                bbox=dict(boxstyle='circle,pad=0.25', fc=color, ec='w', lw=0.8))


def show_policies(rec, n=8, seed=1):
    rules = [('uniform', False, 'uniform'), ('saliency', False, 'saliency'), ('saliency', True, 'saliency + IoR')]
    fig = plt.figure(figsize=(12.5, 6.2))
    for j, (pol, ior, name) in enumerate(rules):
        fix = sample_fixations(rec, n, pol, ior, np.random.default_rng(seed))
        draw_scanpath(fig.add_axes([0.01 + j / 3, 0.3, 0.31, 0.62]), rec, fix, POLICY_COLORS[name], name)
    frame = load_frame(rec)
    for i, (r, c) in enumerate(fix):                        # the glimpses of the last (IoR) scanpath
        a = fig.add_axes([0.06 + i * 0.112, 0.03, 0.1, 0.2])
        a.imshow(glimpse(frame, r, c).permute(1, 2, 0)); a.set_xticks([]); a.set_yticks([])
        for s in a.spines.values():
            s.set_color(POLICY_COLORS['saliency + IoR']); s.set_visible(True)
        a.set_title(f'glimpse {i + 1}', fontsize=8, color=INK2)
    plt.show()


show_policies(IMAGES[IMAGE])

### Coding Exercise A.1: how much of the image do eight glimpses show?

*Exercise objective:* measure which share of the image a scanpath has seen.

The model only knows what falls inside its glimpses. Complete `coverage(rec, fix)`:

1. `seen` starts as a 512 x 512 array of `False`. For every fixation `(r, c)`, set the pixels of the 128 x 128
   window around it to `True`.
2. Return the share of the image's pixels that are `True` in `seen`. The array `image_pixels(rec)` is a 512 x 512 array that is `True` exactly on the image.

Then uncomment the last line: it compares the three rules over 100 images.

In [ ]:
RULES = [('uniform', False, 'uniform'), ('saliency', False, 'saliency'), ('saliency', True, 'saliency + IoR')]


def image_pixels(rec):
    """512 x 512 array: True where the frame shows the image, False on the black bars."""
    return np.kron(content_mask(rec), np.ones((CELL, CELL))).astype(bool)


def coverage(rec, fix):
    """Share of the image's pixels that are inside at least one glimpse. fix: [n, 2] fixations (row, col) in pixels."""
    #################################################
    ## TODO : fill in the two ... and remove the next line
    raise NotImplementedError('Student exercise: coverage')
    #################################################
    seen = np.zeros((FRAME, FRAME), dtype=bool)
    for r, c in fix:
        seen[..., ...] = True     # Hint: rows r - 64 to r + 64, columns c - 64 to c + 64; use max(0, ...) for the start
    return ...                    # Hint: seen[image_pixels(rec)] keeps the image pixels; the mean of True/False is the share of True


def compare_coverage(n=8):
    for pol, ior, name in RULES:
        v = [coverage(r, sample_fixations(r, n, pol, ior, np.random.default_rng(i))) for i, r in enumerate(IMAGES[:100])]
        print(f'{name:15s} {n} glimpses show {np.mean(v):.0%} of the image')


# Uncomment to test
# compare_coverage()

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def coverage(rec, fix):
    seen = np.zeros((FRAME, FRAME), dtype=bool)
    for r, c in fix:
        seen[max(0, r - 64):r + 64, max(0, c - 64):c + 64] = True
    return seen[image_pixels(rec)].mean()
```

Plain saliency with no IoR shows the least of the image as it keeps going back to the same few salient places, so the glimpses overlap.
IoR forbids going back, so each new glimpse adds new pixels to coverage. Uniform glimpses also spread out, but nothing stops
two of them from landing close to each other. Interestingly, the uniform policy shows less of the image than saliency with IoR.

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('coverage')
compare_coverage()

**How salient is the k-th glimpse?** For 100 images, we measure the average saliency inside each glimpse
(1 = the image's average, i.e. a random glimpse location on the image) along 16 fixations, under each rule.

In [ ]:
def glimpse_saliency(rec, fix):
    s = SALIENCY[rec['sal_row']].astype(np.float32)
    s = s / s[content_mask(rec)].mean()
    half = FOVEA // 2 // CELL
    vals = []
    for r, c in fix:
        r, c = r // CELL, c // CELL
        win = s[max(0, r - half):r + half, max(0, c - half):c + half]
        vals.append(win.sum() / (2 * half) ** 2)               # outside the frame counts as 0
    return np.array(vals)


K = 16
for pol, ior, name in [('uniform', False, 'uniform'), ('saliency', False, 'saliency'), ('saliency', True, 'saliency + IoR')]:
    v = np.mean([glimpse_saliency(r, sample_fixations(r, K, pol, ior, np.random.default_rng(i)))
                 for i, r in enumerate(IMAGES[:100])], 0)
    plt.plot(np.arange(1, K + 1), v, marker='o', ms=4, color=POLICY_COLORS[name], label=name)
plt.axhline(1, color=INK3, lw=0.8, ls='--')
plt.xlabel('fixation k'); plt.ylabel('saliency inside the glimpse\n(1 = a random location)'); plt.legend()
plt.show()

### Think! A.1

Why does the IoR curve sink below the plain saliency curve, and even below 1, after about nine
glimpses? What does that mean for the images a model trained with IoR sees late in a sequence?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Plain saliency keeps returning to the same few high-saliency regions, so every glimpse is salient. With IoR each
glimpse removes its 128 x 128 window (1/16 of the frame) forever. After about nine glimpses roughly half the
frame has been covered, and that half held most of the saliency, so what is left is below average saliency and late
glimpses land on background. A model trained with IoR therefore sees many plain, low-content glimpses late in
each sequence, which matters for what it learns to predict.

</details>

## B. seq-JEPA: learning to predict the upcoming glimpses

The seq-JEPA model we train here has three parts:

1. an **encoder** (EfficientNet-B0) that is a convolutional neural network that processes the incoming glimpse image and converts it into a 1280-dimensional vector which is the visual representation of the glimpse. 
2. an **LSTM** that keeps a running memory of the glimpses so far. Each glimpse's visual representation goes in together with the
   saccade that brought the eye there, a copy of the motor command leading to that observation;
3. a **predictor** that takes the LSTM hidden state and the *next* saccade and predicts the representation of the next glimpse.

Training needs no labels and is fully self-supervised. The model is asked to make its prediction match the actual next glimpse. The
targets come from a slowly updated copy of the encoder (an exponential moving average, EMA), a standard trick
that keeps the representations from collapsing. Saccadic actions are (row, column) displacement vectors.

We trained four versions for 300 epochs on 113k COCO images, with 9 glimpses per sequence (after each glimpse, the model predicts the next one). The three first models have different glimpse rules: uniform, saliency, saliency + IoR. The fourth model uses the saliency + IoR policy but has **no actions** and never receives the
saccades as its action input.

In [ ]:
class SeqJEPA(nn.Module):
    def __init__(self, use_action=True):
        super().__init__()
        self.encoder = torchvision.models.efficientnet_b0()
        self.encoder.classifier[1] = nn.Identity()                  # 1280-d glimpse representation
        self.target_encoder = torchvision.models.efficientnet_b0()  # the slow EMA copy: prediction targets
        self.target_encoder.classifier[1] = nn.Identity()
        self.act_emb = nn.Linear(2, 128)                            # saccade -> 128-d action embedding
        self.seq_model = nn.LSTM(1280 + 128, 1408, batch_first=True)
        self.predictor = nn.Sequential(nn.Linear(1408 + 128, 1024, bias=False), nn.ReLU(), nn.Linear(1024, 1280))
        self.use_action = use_action

    def action(self, sacc):                                        # [..., 2] saccade / 512 -> [..., 128]
        a = self.act_emb(sacc)
        return a if self.use_action else torch.zeros_like(a)

    def encode(self, g):                                           # [B, T, 3, 128, 128] -> [B, T, 1280]
        return self.encoder(g.flatten(0, 1)).view(*g.shape[:2], -1)

    def integrate(self, z, fix):
        """Glimpse representations [B, T, 1280] and fixations [B, T, 2] (frame px) -> LSTM state after each glimpse.
        Glimpse t goes in with the saccade that brought the eye to it (zero for the first)."""
        incoming = torch.cat((torch.zeros_like(fix[:, :1]), fix[:, 1:] - fix[:, :-1]), 1) / FRAME
        h, _ = self.seq_model(torch.cat((z, self.action(incoming)), -1))
        return h

    def predict(self, h, next_sacc):                               # memory + next saccade (px) -> next representation
        return self.predictor(torch.cat((h, self.action(next_sacc / FRAME)), -1))

### Coding Exercise B.1: the self-supervised loss

*Exercise objective:* write the loss that seq-JEPA is trained with.

Take a batch of B sequences of T glimpses each. We have:

- `z`: the encoder's representation of every glimpse, shape [B, T, 1280];
- `targets`: the representation of every glimpse from the *target encoder*, a slowly changing copy of the encoder
  (Exercise B.2), shape [B, T, 1280]. These are what the model has to predict;
- `fix`: the fixations, (row, column) in pixels, shape [B, T, 2].

After each glimpse t (t = 1 ... T-1), the model predicts the representation of glimpse t+1. It uses
the LSTM state after glimpse t (`h`) and the saccade from fixation t to fixation t+1, which
is the difference of the two fixations. You need to:

1. compute the saccades `sacc`, shape [B, T-1, 2];
2. compare each prediction with the target representation of the glimpse it predicts. We use the *cosine similarity* as our comparison metric which is
   1 when two vectors point in the same direction and 0 when they are in opposite directions. The loss is
   `1 - cosine similarity`, averaged over all predictions, so a perfect prediction gives a loss of 0.

Hint: `fix[:, 1:]` are fixations 2 ... T and `fix[:, :-1]` are fixations 1 ... T-1. The same slicing works for
`targets`.

In [ ]:
def next_step_loss(model, z, targets, fix):
    """Predict the representation of glimpse t+1 from the memory after glimpse t and the saccade t -> t+1; return 1 - cos."""
    #################################################
    ## TODO: fill in the two ... and remove the next line
    raise NotImplementedError('Student exercise: next_step_loss')
    #################################################
    h = model.integrate(z[:, :-1], fix[:, :-1])      # LSTM state after glimpses 1 ... T-1: [B, T-1, 1408]
    sacc = ...                                       # saccades 1 -> 2, ..., T-1 -> T: [B, T-1, 2]
    pred = model.predict(h, sacc)                    # predicted representations of glimpses 2 ... T: [B, T-1, 1280]
    return ...                                       # Hint: F.cosine_similarity(a, b, dim=-1), then .mean()


# Uncomment to test: with random representations the loss should be close to 1
# print(next_step_loss(SeqJEPA(), torch.randn(2, 5, 1280), torch.randn(2, 5, 1280), torch.rand(2, 5, 2) * FRAME))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def next_step_loss(model, z, targets, fix):
    h = model.integrate(z[:, :-1], fix[:, :-1])
    sacc = fix[:, 1:] - fix[:, :-1]
    pred = model.predict(h, sacc)
    return 1 - F.cosine_similarity(pred, targets[:, 1:], dim=-1).mean()
```

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('next_step_loss')
print(next_step_loss(SeqJEPA(), torch.randn(2, 5, 1280), torch.randn(2, 5, 1280), torch.rand(2, 5, 2) * FRAME))

### Coding Exercise B.2: the slow target encoder

*Exercise objective:* update the target encoder as a moving average of the encoder.

Where do the targets come from? If the exact same encoder that is trained also made the targets, the model could lower the
loss by cheating, i.e., it could give every glimpse the same representation, and every prediction would be perfect with a loss of 0. This failure is called representational
*collapse*. A standard fix for representational collapse is a second copy of the encoder, the **target encoder** that is not trained by
gradient descent. Instead, after every training step, each of its weights moves a small step towards the same
weight of the trained encoder:

target weight = ema x target weight + (1 - ema) x encoder weight, with ema = 0.99.

This is an *exponential moving average* (EMA): the target encoder follows the encoder, but slowly. Complete
`ema_update`.

In [ ]:
@torch.no_grad()
def ema_update(encoder, target_encoder, ema=0.99):
    """Move every weight of target_encoder a step towards the same weight of encoder."""
    #################################################
    ## TODO: fill in the ... and remove the next line
    raise NotImplementedError('Student exercise: ema_update')
    #################################################
    for p, q in zip(encoder.parameters(), target_encoder.parameters()):   # p: encoder weight, q: target weight
        q.copy_(...)                                                        # Hint: the formula above, with q and p


# Uncomment to test: a target weight of 0 that moves towards 1 with ema = 0.9 should become 0.1
# a, b = nn.Linear(1, 1, bias=False), nn.Linear(1, 1, bias=False)
# nn.init.ones_(a.weight); nn.init.zeros_(b.weight)
# ema_update(a, b, ema=0.9); print(b.weight.item())

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
@torch.no_grad()
def ema_update(encoder, target_encoder, ema=0.99):
    for p, q in zip(encoder.parameters(), target_encoder.parameters()):
        q.copy_(ema * q + (1 - ema) * p)
```

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('ema_update')
a, b = nn.Linear(1, 1, bias=False), nn.Linear(1, 1, bias=False)
nn.init.ones_(a.weight); nn.init.zeros_(b.weight)
ema_update(a, b, ema=0.9); print(f'{b.weight.item():.2f}')

### Watch it learn

We perform a few training steps starting from random weights. Each step uses 4
sequences of 5 glimpses. On a laptop CPU one step takes a few seconds; the real models were trained for 300 epochs
on GPUs. The training loop below uses your `next_step_loss` (B.1) and `ema_update` (B.2).

In [ ]:
def training_batches(n_seq, T, policy='uniform', ior=False, seed=0):
    g = np.random.default_rng(seed)
    while True:
        recs = [IMAGES[i] for i in g.choice(len(IMAGES), n_seq, replace=False)]
        fixes = [sample_fixations(r, T, policy, ior, g) for r in recs]
        x = torch.stack([glimpses(load_frame(r), f) for r, f in zip(recs, fixes)])
        yield normalize(x), torch.as_tensor(np.stack(fixes), dtype=torch.float32)


def train_from_scratch(steps=8, n_seq=4, T=5, lr=1e-3, ema=0.99):
    model = SeqJEPA()
    for m in model.modules():                                      # as in our runs: no dropout, no stochastic depth
        if isinstance(m, (nn.Dropout, torchvision.ops.StochasticDepth)):
            m.p = 0.0
    model.target_encoder.load_state_dict(model.encoder.state_dict())
    model.target_encoder.requires_grad_(False)
    opt = torch.optim.AdamW([p for n, p in model.named_parameters() if not n.startswith('target_')], lr=lr)
    data = training_batches(n_seq, T, seed=SEED)
    for step in range(1, steps + 1):
        t0 = time.time()
        x, fix = next(data)
        with torch.no_grad():
            targets = model.target_encoder(x.flatten(0, 1)).view(*x.shape[:2], -1)
        loss = next_step_loss(model, model.encode(x), targets, fix)
        opt.zero_grad(); loss.backward(); opt.step()
        ema_update(model.encoder, model.target_encoder, ema)       # the target encoder follows slowly
        print(f'step {step}: loss {loss.item():.3f}  ({time.time() - t0:.1f} s)')
    return model


_ = train_from_scratch()

### The trained models

Below we load two of the final checkpoints after 300 epochs.

In [ ]:
def load_seqjepa(name):
    sd = torch.load(os.path.join(DATA_DIR, 'weights', f'{name}.pt'), map_location='cpu')
    info = json.load(open(os.path.join(DATA_DIR, 'weights', 'manifest.json')))[name]
    model = SeqJEPA(use_action=info['action_pairing'] != 'none')
    model.load_state_dict({k: v.float() for k, v in sd.items()})
    return model.eval().requires_grad_(False), info


seqjepa, info = load_seqjepa('seqjepa')
noact, info_noact = load_seqjepa('seqjepa_noact')
for name, i in (('seq-JEPA', info), ('seq-JEPA (no action)', info_noact)):
    print(f"{name:22s} {i['epoch']} epochs, glimpses: {i['coco_fixation_sampler']}{' + IoR' if i['ior'] else ''}, "
          f"saccades as input: {i['action_pairing'] != 'none'}")
print(f'{sum(p.numel() for p in seqjepa.parameters()) / 1e6:.1f} M parameters')

**Can it predict what it will see where it looks?** We let each model look at 12 uniformly placed glimpses per
image. After glimpse t (t = 4 ... 7), we give the predictor the true next saccade and ask which of five
candidate glimpses it expects: the true next one, or one of the four after it (all from the same image, so
they are hard to tell apart without knowing *where* the eye goes). The chance performance is 20%.

The cell runs both loaded models on 30 images.

In [ ]:
@torch.no_grad()
def look(model, recs, n=12, seed=0):
    """Per image: n uniform fixations, the LSTM state after each glimpse, target representations of glimpses 5..n."""
    g, out = np.random.default_rng(seed), []
    for rec in recs:
        fix = sample_fixations(rec, n, 'uniform', False, g)
        x = normalize(glimpses(load_frame(rec), fix))
        h = model.integrate(model.encoder(x)[None], torch.as_tensor(fix, dtype=torch.float32)[None])[0]
        tgt = torch.zeros(n, 1280)
        tgt[4:] = F.normalize(model.target_encoder(x[4:]), dim=-1)
        out.append({'fix': fix, 'h': h, 'target': tgt})
    return out


@torch.no_grad()
def retrieval(model, seqs, n_cand=5):
    hits = []
    for s in seqs:
        for t in range(3, len(s['fix']) - n_cand):               # memory after glimpse t+1 (0-based t)
            cand = np.arange(t + 1, t + 1 + n_cand)              # candidate 0 is the true next glimpse
            pred = model.predict(s['h'][t], torch.as_tensor(s['fix'][cand[0]] - s['fix'][t], dtype=torch.float32))
            hits.append(int((s['target'][cand] @ F.normalize(pred, dim=-1)).argmax() == 0))
    return np.mean(hits)


t0 = time.time()
SEQS = {'seq-JEPA': look(seqjepa, IMAGES[:30]), 'seq-JEPA (no action)': look(noact, IMAGES[:30])}
MODELS = {'seq-JEPA': seqjepa, 'seq-JEPA (no action)': noact}
for name, seqs in SEQS.items():
    print(f'{name:22s} picks the right next glimpse {100 * retrieval(MODELS[name], seqs):.0f}% of the time (chance 20%)')
print(f'({time.time() - t0:.0f} s)')

**Seeing the predictions: nearest neighbours.** Five candidates make an easy test. Here we give the model a harder one, with
the seq-JEPA model trained on **saliency + IoR** glimpses (the scanpaths of Section A, right panel).

For each of 30 images we sample a saliency + IoR scanpath of 10 glimpses. The target representations of all of them, 300
glimpses in total, go into one *pool* of glimpses. For each image, the model sees glimpses 1 to 4 (purple). Then the
predictor gets the LSTM memory and the next saccade (amber arrow) and predicts the representation of glimpse 5. We rank
the whole pool by cosine similarity to that prediction and show the 3 nearest glimpses. The true glimpse 5 is
in the pool, and so are 9 other glimpses of the same image, which look a lot like it.

As a control, we also give the predictor a saccade of 0, which means "look at the same place again". The right
answer is then glimpse 4 itself.

The cell takes about 20 seconds. Change `rows` in `show_neighbours(POOL, rows=...)` to see other images (0 to 29).

In [ ]:
from matplotlib.patches import FancyArrowPatch

salior, _ = load_seqjepa('seqjepa_salior')          # seq-JEPA trained on saliency + IoR glimpses
CTX, N_FIX, AMBER, HIT = 4, 10, '#eda100', '#2b8a3e'   # 4 context glimpses; 10 glimpses per image in the pool


@torch.no_grad()
def build_pool(model, recs, seed=0):
    """Per image: a saliency + IoR scanpath, the memory after CTX glimpses, and the target representations of all its glimpses."""
    g, seqs = np.random.default_rng(seed), []
    for rec in recs:
        fix = sample_fixations(rec, N_FIX, 'saliency', True, g)
        x = normalize(glimpses(load_frame(rec), fix))
        h = model.integrate(model.encoder(x[:CTX])[None], torch.as_tensor(fix[:CTX], dtype=torch.float32)[None])
        seqs.append({'rec': rec, 'fix': fix, 'h': h[0, -1], 'target': F.normalize(model.target_encoder(x), dim=-1)})
    return seqs


@torch.no_grad()
def nearest(model, seqs, i, sacc=None):
    """Predict glimpse CTX + 1 of image i from its memory and the saccade; rank every glimpse in the pool."""
    s = seqs[i]
    sacc = s['fix'][CTX] - s['fix'][CTX - 1] if sacc is None else sacc
    pred = F.normalize(model.predict(s['h'], torch.as_tensor(sacc, dtype=torch.float32)), dim=-1)
    sim = torch.cat([q['target'] for q in seqs]) @ pred      # pool entry p = glimpse p % N_FIX of image p // N_FIX
    order = sim.argsort(descending=True).numpy()
    return order, sim[order].numpy()


def show_neighbours(seqs, rows=(0, 1, 2, 3)):
    W, R, IMG, G, STEP = 12.8, 2.5, 2.3, 1.05, 1.15                 # sizes in inches
    fig = plt.figure(figsize=(W, R * len(rows) + 0.5))
    H = fig.get_figheight()
    box = lambda x, y, w, h: fig.add_axes([x / W, y / H, w / W, h / H])
    xs = {'ctx': 2.55, 'next': 7.85, 'nn': 9.25}
    for j, (x, text) in enumerate([(0.1, 'the image'), (xs['ctx'], f'the {CTX} glimpses it has seen'),
                                   (xs['next'] - 0.4, 'the true next glimpse'),
                                   (xs['nn'], f'nearest neighbours of the prediction, among {len(seqs) * N_FIX} glimpses')]):
        fig.text(x / W, 1 - 0.3 / H, text, fontsize=10, color=INK2, va='center')
    for ri, i in enumerate(rows):
        s, y = seqs[i], H - 0.5 - R * (ri + 1) + (R - IMG) / 2
        yg = y + (IMG - G) / 2
        frame, order, sim = load_frame(s['rec']), *nearest(salior, seqs, i)
        ax = box(0.1, y, IMG, IMG)
        draw_scanpath(ax, s['rec'], s['fix'][:CTX], POLICY_COLORS['saliency + IoR'], saliency=False)
        (r0, c0), (r1, c1) = s['fix'][CTX - 1], s['fix'][CTX]
        ax.add_patch(Rectangle((c1 - FOVEA / 2, r1 - FOVEA / 2), FOVEA, FOVEA, fill=False, ec=AMBER, lw=2, ls='--'))
        ax.annotate('', (c1, r1), (c0, r0), arrowprops=dict(arrowstyle='-|>', color=AMBER, lw=2, shrinkA=8, shrinkB=0))
        for t in range(CTX):
            a = box(xs['ctx'] + t * STEP, yg, G, G)
            a.imshow(glimpse(frame, *s['fix'][t]).permute(1, 2, 0)); a.set_xticks([]); a.set_yticks([])
            plt.setp(list(a.spines.values()), visible=True, color=POLICY_COLORS['saliency + IoR'], lw=1.5)
            a.set_xlabel(f'glimpse {t + 1}', fontsize=8, color=INK2)
        fig.add_artist(FancyArrowPatch(((xs['ctx'] + CTX * STEP) / W, (yg + G / 2) / H), ((xs['next'] - 0.08) / W, (yg + G / 2) / H),
                                       transform=fig.transFigure, arrowstyle='-|>', mutation_scale=14, color=AMBER, lw=2))
        fig.text((xs['ctx'] + CTX * STEP + 0.33) / W, (yg + G / 2 + 0.12) / H, f'saccade\n({r1 - r0:+d}, {c1 - c0:+d})',
                 fontsize=8, color=INK2, ha='center', va='bottom')
        a = box(xs['next'], yg, G, G)
        a.imshow(glimpse(frame, r1, c1).permute(1, 2, 0)); a.set_xticks([]); a.set_yticks([])
        plt.setp(list(a.spines.values()), visible=True, color=AMBER, lw=2, ls='--')
        a.set_xlabel(f'glimpse {CTX + 1}', fontsize=8, color=INK2)
        for n in range(3):
            p = order[n]
            src, k = seqs[p // N_FIX], p % N_FIX
            hit = p == i * N_FIX + CTX
            a = box(xs['nn'] + n * STEP, yg, G, G)
            a.imshow(glimpse(load_frame(src['rec']), *src['fix'][k]).permute(1, 2, 0)); a.set_xticks([]); a.set_yticks([])
            plt.setp(list(a.spines.values()), visible=True, color=HIT if hit else INK3, lw=3 if hit else 1)
            a.set_title(f'#{n + 1}   cos {sim[n]:.2f}', fontsize=8, color=INK2)
            where = 'the true\nglimpse' if hit else (f'same image,\nglimpse {k + 1}' if p // N_FIX == i else 'another\nimage')
            a.set_xlabel(where, fontsize=8, color=HIT if hit else INK2, fontweight='bold' if hit else 'normal')
    plt.show()


t0 = time.time()
POOL = build_pool(salior, IMAGES[:30])
ranks = np.array([list(nearest(salior, POOL, i)[0]).index(i * N_FIX + CTX) for i in range(len(POOL))])
print(f'the true glimpse {CTX + 1} is the nearest neighbour for {np.mean(ranks < 1):.0%} of the images and in the top 3 '
      f'for {np.mean(ranks < 3):.0%} (by chance: {1 / (len(POOL) * N_FIX):.1%} and {3 / (len(POOL) * N_FIX):.0%})')
same = np.mean([nearest(salior, POOL, i, np.zeros(2))[0][0] == i * N_FIX + CTX - 1 for i in range(len(POOL))])
print(f'with a saccade of 0 ("look at the same place again"), the nearest neighbour is glimpse {CTX} itself for '
      f'{same:.0%} of the images ({time.time() - t0:.0f} s)')
show_neighbours(POOL)

### Coding Exercise B.3: steer the prediction

*Exercise objective:* check that the prediction depends on the saccade it is given.

If the model really uses the saccade, we should be able to *steer* its prediction. In `retrieval` above, the
predictor was always given the true saccade, ie the one to candidate 0 which is the true next glimpse. Complete `steering`, a copy of `retrieval`
with one change: for each test, pick a random candidate `j` (1 to 4), give the predictor the saccade to
candidate `j` instead, and count a hit when the model picks candidate `j`.

Before you run the code, what do you expect for seq-JEPA, and for the model without actions?

In [ ]:
@torch.no_grad()
def steering(model, seqs, n_cand=5, seed=0):
    """Like retrieval, but the predictor is told that the eye goes to a random candidate j; a hit = the model picks j as the next fixation."""
    #################################################
    ## TODO: fill in the three ... and remove the next line
    raise NotImplementedError('Student exercise: steering')
    #################################################
    g, hits = np.random.default_rng(seed), []
    for s in seqs:
        for t in range(3, len(s['fix']) - n_cand):
            cand = np.arange(t + 1, t + 1 + n_cand)            # the 5 candidate glimpses (positions in the sequence)
            j = ...                                            # a random candidate, 1 to n_cand - 1. Hint: g.integers(low, high)
            sacc = ...                                         # the saccade from fixation t to the fixation of candidate j
            pred = model.predict(s['h'][t], torch.as_tensor(sacc, dtype=torch.float32))
            picked = (s['target'][cand] @ F.normalize(pred, dim=-1)).argmax()   # the candidate closest to the prediction
            hits.append(int(picked == ...))
    return np.mean(hits)


# Uncomment to test
# for name, seqs in SEQS.items():
#     print(f'{name:22s} follows the saccade {100 * steering(MODELS[name], seqs):.0f}% of the time (chance 20%)')

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
@torch.no_grad()
def steering(model, seqs, n_cand=5, seed=0):
    g, hits = np.random.default_rng(seed), []
    for s in seqs:
        for t in range(3, len(s['fix']) - n_cand):
            cand = np.arange(t + 1, t + 1 + n_cand)
            j = g.integers(1, n_cand)                            # pretend the eye goes to candidate j
            sacc = s['fix'][cand[j]] - s['fix'][t]
            pred = model.predict(s['h'][t], torch.as_tensor(sacc, dtype=torch.float32))
            hits.append(int((s['target'][cand] @ F.normalize(pred, dim=-1)).argmax() == j))
    return np.mean(hits)
```

seq-JEPA follows the saccade it is given as its prediction is about the place it is told it will look.
The model without actions cannot be steered, because it never sees saccades: it can only predict what a typical
next glimpse of this image looks like.

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('steering')
for name, seqs in SEQS.items():
    print(f'{name:22s} follows the saccade {100 * steering(MODELS[name], seqs):.0f}% of the time (chance 20%)')

## C. What is in its memory?

A **linear probe** asks what information can be read out of a representation with a single linear map. Here, we
freeze the model, collect its LSTM state after some glimpses, and fit a linear regression (or classifier) from
the state to a quantity we care about. Good test performance means the quantity is indeed encoded in the representations and can be linearly recovered.

### Coding Exercise C.1: where am I?

*Exercise objective:* test whether the LSTM keeps track of where the eye is.

If the LSTM adds up the saccades it receives, it should know where the eye is relative to where it started (*path integration*). We test this with the sequences from Section B (30 images, 12
glimpses each):

- `X` has one row per glimpse after the first: the LSTM state after glimpses 2 ... 12 of the first image, then
  of the second image, and so on (11 rows per image).
- You should fill in `Y`, with one row per row of `X`, in the same order. It is the vector from the current fixation back to
  the first fixation, normalized by the frame size (512). `s['fix']` holds the 12 fixations of one image, shape
  [12, 2].

`ridge_r2` (given) fits a linear regression from `X` to `Y` on 70% of the images and reports R2 on the other 30%
(1 = perfect, 0 = no better than always guessing the average). Compare the two models. Which one can tell
where it is, and why?

In [ ]:
def ridge_r2(Xtr, Ytr, Xte, Yte, lams=10.0 ** np.arange(-1, 5)):
    """Test R2 of a ridge regression on standardized features (lambda chosen on held-out training rows)."""
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-3
    A, B = (Xtr - mu) / sd, (Xte - mu) / sd
    fit = lambda X, Y, l: (np.linalg.solve(X.T @ X + l * np.eye(X.shape[1]), X.T @ (Y - Y.mean(0))), Y.mean(0))
    r2 = lambda w, b, X, Y: 1 - ((Y - X @ w - b) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()
    n = int(0.8 * len(A))
    lam = max(lams, key=lambda l: r2(*fit(A[:n], Ytr[:n], l), A[n:], Ytr[n:]))
    return r2(*fit(A, Ytr, lam), B, Yte)


def path_integration(seqs):
    """R2 of a linear readout, from the LSTM state, of the way back to the first fixation."""
    #################################################
    ## TODO: fill in the ... and remove the next line
    raise NotImplementedError('Student exercise: path_integration')
    #################################################
    X = np.concatenate([s['h'][1:].numpy() for s in seqs])      # LSTM state after glimpses 2 ... 12, image by image
    Y = ...                                                      # Hint: s['fix'][:1] - s['fix'][1:] for each image, then np.concatenate
    n_tr = int(0.7 * len(seqs)) * (len(seqs[0]['fix']) - 1)      # the rows of the first 70% of the images
    return ridge_r2(X[:n_tr], Y[:n_tr], X[n_tr:], Y[n_tr:])


# Uncomment to test
# for name, seqs in SEQS.items():
#     print(f'{name:22s} R2 of the way back to the first fixation: {path_integration(seqs):.2f}')

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def path_integration(seqs):
    X = np.concatenate([s['h'][1:].numpy() for s in seqs])
    Y = np.concatenate([(s['fix'][:1] - s['fix'][1:]) / FRAME for s in seqs])
    n_tr = int(0.7 * len(seqs)) * (len(seqs[0]['fix']) - 1)
    return ridge_r2(X[:n_tr], Y[:n_tr], X[n_tr:], Y[n_tr:])
```

With the efference copy (the action copy given to LSTM), the LSTM adds up the saccades and the way back can be read out linearly.
Without actions, the state only holds what the glimpses looked like, which says little about where they were.

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('path_integration')
for name, seqs in SEQS.items():
    print(f'{name:22s} R2 of the way back to the first fixation: {path_integration(seqs):.2f}')

### What does it know about the objects?

The same evaluation recipe, with object labels instead of positions. Our probes read from the LSTM state after k
glimpses:

- which of the 80 COCO categories appear in the image (**multi-label mAP**: average precision per category,
  averaged over categories);
- which categories appear in each cell of a 10 x 10 grid over the image (**grid F1**: a coarse segmentation,
  which needs the model to know both *what* and *where*).

Those probes were fit on 113k training images and tested on the 5k COCO val2017 images. Here is the
computation at toy scale, on the 300 images of this pack. With only 200 training images the numbers are noisy;
the point is the procedure.

In [ ]:
EMB_VAL = np.load(os.path.join(DATA_DIR, 'search', 'codes_val.npy'))   # [300, 101, 1280]: 100 cells + centre
CATEGORIES = META['categories']
CELL_CENTRES = np.array([((i // GRID) * (FRAME // GRID) + FRAME // GRID // 2, (i % GRID) * (FRAME // GRID) + FRAME // GRID // 2)
                         for i in range(GRID * GRID)])                     # where the stored representations were taken


def cell_saliency(rec):
    """Saliency mass of each of the 100 action cells."""
    s = SALIENCY[rec['sal_row']].astype(np.float64)
    k = np.arange(SAL_GRID) * GRID // SAL_GRID                             # action cell of each saliency row
    return np.bincount((k[:, None] * GRID + k[None]).ravel(), s.ravel(), GRID * GRID)


def content_cells(rec):
    """The action cells that overlap the image (cells that are only black bar are not allowed)."""
    s = FRAME / max(rec['width'], rec['height'])
    nw, nh = int(round(rec['width'] * s)), int(round(rec['height'] * s))
    pl, pt = (FRAME - nw) // 2, (FRAME - nh) // 2
    e = np.arange(GRID + 1) * FRAME / GRID
    return (((e[1:] > pt) & (e[:-1] < pt + nh))[:, None] & ((e[1:] > pl) & (e[:-1] < pl + nw))[None]).ravel()


def cell_scanpath(rec, k, policy, gen):
    """k cells chosen by a glimpse rule at cell resolution (IoR switches off the 3 x 3 cells around a fixation)."""
    p = (cell_saliency(rec) if policy.startswith('saliency') else np.ones(GRID * GRID)) * content_cells(rec)
    cells = []
    for _ in range(k):
        c = gen.choice(GRID * GRID, p=p / p.sum())
        cells.append(c)
        if policy == 'saliency + IoR':
            r0, c0 = c // GRID, c % GRID
            for r in range(max(0, r0 - 1), min(GRID, r0 + 2)):
                p[r * GRID + max(0, c0 - 1):r * GRID + min(GRID, c0 + 2)] *= 1e-9
    return np.array(cells)


@torch.no_grad()
def states_after(model, rows, k, policy, seed=0):
    """LSTM state after k glimpses at cell centres, one sequence per image."""
    g = np.random.default_rng(seed)
    cells = np.stack([cell_scanpath(IMAGES[r], k, policy, g) for r in rows])
    z = torch.from_numpy(EMB_VAL[rows[:, None], cells].astype(np.float32))
    fix = torch.as_tensor(CELL_CENTRES[cells], dtype=torch.float32)
    return model.integrate(z, fix)[:, -1].numpy()


def labels(rows):
    y = np.zeros((len(rows), len(CATEGORIES)))
    for i, r in enumerate(rows):
        for inst in IMAGES[r]['instances']:
            y[i, CATEGORIES.index(inst['category'])] = 1
    return y


def average_precision(y, score):
    y = y[np.argsort(-score)]
    precision = np.cumsum(y) / np.arange(1, len(y) + 1)
    return (precision * y).sum() / y.sum()


def category_map(X, Y, n_train=200, lam=100.0):
    """Ridge probe from states to the multi-hot category vector; mAP on the held-out images."""
    mu, sd = X[:n_train].mean(0), X[:n_train].std(0) + 1e-3
    A, B = (X[:n_train] - mu) / sd, (X[n_train:] - mu) / sd
    W = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ (Y[:n_train] - Y[:n_train].mean(0)))
    S, Yte = B @ W, Y[n_train:]
    keep = Yte.sum(0) >= 2                                          # categories present at least twice
    return np.mean([average_precision(Yte[:, c], S[:, c]) for c in np.flatnonzero(keep)])


rows = np.arange(len(IMAGES))
Y = labels(rows)
for policy in ('uniform', 'saliency', 'saliency + IoR'):
    print(f'seq-JEPA, 8 glimpses by {policy:15s} mAP {category_map(states_after(seqjepa, rows, 8, policy), Y):.3f}')

The full-scale results for all four models are below (mean ± sd over 3 training seeds). Columns: the glimpse
rule used *at test time*, the same fixations for every model. Curves: the rule each model was *trained* with.

On the right of each panel (grey band) are the same probes for a model that looks at the **full image** at
once, in a single pass of its encoder:

- coloured dots: each seq-JEPA model's own encoder, given the whole 512 x 512 image instead of a glimpse
  (mean over 3 seeds);
- open triangle and diamond: SimCLR and BYOL, two standard self-supervised models with the same encoder,
  trained for 300 epochs on the same images, on two randomly augmented copies of the whole image (one seed
  each).

The x-axis is placed by cost: one pass over the full image costs about as many operations as 14 glimpses (the
second line under the bottom panels gives the cost in GFLOPs).

In [ ]:
from matplotlib.lines import Line2D

PROBES = json.load(open(os.path.join(DATA_DIR, 'results', 'probes.json')))
TRAIN_COLORS = {'uniform': POLICY_COLORS['uniform'], 'saliency + IoR': POLICY_COLORS['saliency + IoR'],
                'saliency, no IoR': POLICY_COLORS['saliency'], 'no action': '#eb6834'}
TRAIN_LABELS = {'uniform': 'uniform', 'saliency + IoR': 'saliency + IoR', 'saliency, no IoR': 'saliency', 'no action': 'no action'}
OTHERS = {'SimCLR': '^', 'BYOL': 'D'}                      # full-image models trained on whole images (one seed)


def mean_sd(v):
    """Mean and sd (n - 1) over training seeds, the first axis."""
    v = np.array(v)
    return v.mean(0), v.std(0, ddof=1) if len(v) > 1 else 0 * v[0]


def plot_probes(probes):
    evals = [('saliency_ior', 'test glimpses: saliency + IoR'), ('saliency_noior', 'test glimpses: saliency'),
             ('uniform', 'test glimpses: uniform')]
    metrics = [('mAP', 'multi-label mAP'), ('grid', 'grid F1')]
    ks = [4, 6, 8, 12]
    gf = probes['gflops']
    step = gf['glimpse'] + gf['lstm_step']                    # GFLOPs per glimpse: encoder + one LSTM step
    x_full = gf['full_image'] / step                         # the full image costs as much as this many glimpses
    fig, axes = plt.subplots(2, 3, figsize=(13, 7.8), sharey='row')
    for ri, (met, mlabel) in enumerate(metrics):
        for ci, (ev, title) in enumerate(evals):
            ax = axes[ri, ci]
            ax.grid(axis='y', color=GRIDC, lw=0.8)
            ax.axvspan(x_full - 0.8, x_full + 0.9, color=GRIDC, alpha=0.6, lw=0)
            for j, (trained, color) in enumerate(TRAIN_COLORS.items()):
                m, s = mean_sd([[r[ev][str(k)][met] for k in ks] for r in probes['lstm'][trained].values()])
                ax.fill_between(ks, m - s, m + s, color=color, alpha=0.15, lw=0)
                ax.plot(ks, m, marker='o', ms=4.5, lw=2, color=color)
                m = np.mean([r[met] for r in probes['full_image'][trained].values()])
                ax.plot(x_full - 0.5 + 0.17 * j, m, 'o', ms=6.5, color=color, mec='w')
            for j, (name, marker) in enumerate(OTHERS.items()):
                ax.plot(x_full + 0.35 + 0.3 * j, probes['full_image'][name]['42'][met], marker, ms=6.5, color=INK2,
                        mfc='w', mew=1.5)
            cost = [f'\n{k * step:.1f}' if ri == 1 else '' for k in ks] + [f'\n{gf["full_image"]:.1f}' if ri == 1 else '']
            ax.set_xticks(ks + [x_full], [f'{k}{c}' for k, c in zip(ks + ['full'], cost)])
            ax.set_xlim(3, x_full + 1.2)
            if ri == 0:
                ax.set_title(title)
            if ri == 1:
                ax.set_xlabel('glimpses k   (second line: GFLOPs per image)')
            if ci == 0:
                ax.set_ylabel(mlabel)
    handles = [Line2D([], [], color=c, lw=2, marker='o', ms=4.5, label=f'trained on {TRAIN_LABELS[t]}')
               for t, c in TRAIN_COLORS.items()]
    handles += [Line2D([], [], ls='', marker='o', ms=5, color=INK3, label='its own encoder, full image')]
    handles += [Line2D([], [], ls='', marker=mk, ms=6.5, color=INK2, mfc='w', mew=1.5, label=f'{n}, full image')
                for n, mk in OTHERS.items()]
    fig.legend(handles=handles, loc='upper center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, 1.0))
    plt.tight_layout(rect=(0, 0, 1, 0.92)); plt.show()


plot_probes(PROBES)

### Think! C.1

Three things to explain from this figure:

1. For every model, saliency + IoR at test time gives the best grid F1. Why would the test-time rule matter?
2. The model without actions has the *best* mAP but the *worst* grid F1. What does the efference copy (action) add that
   mAP does not need but grid F1 does?
3. Compare seq-JEPA trained on uniform glimpses (blue) after 8 saliency + IoR glimpses (left column) with the dots on the right. Is reading the image glimpse
   by glimpse better or worse than encoding the whole image? At what cost?

<details>
<summary><b>Solution</b> (click to expand)</summary>

1. The test-time rule decides what the model gets to see. Saliency + IoR puts the glimpses on objects and
   never on the same place twice, so after k glimpses more of the image's objects have been seen. Every model
   benefits from this.
2. mAP only asks which categories are present anywhere, so a bag of glimpse representations is enough, and the no-action
   model, which devotes its whole memory to appearance, is good at it. Grid F1 asks where each category is,
   which needs to know where each glimpse was taken: that is exactly what the saccade input provides.

| | seq-JEPA, 8 glimpses | its encoder, full image | SimCLR | BYOL |
|---|---|---|---|---|
| mAP | 0.464 | 0.434 | 0.461 | 0.399 |
| grid | 0.584 | 0.576 | 0.584 | 0.569 |

3. After 8 glimpses, seq-JEPA matches SimCLR and does better than its own encoder on the full image and than BYOL
   (SimCLR and BYOL were trained on whole images; one seed each). It does so for about 57% of the operations
   of one full-image pass. (A caveat: seq-JEPA's encoder only ever saw 128 x 128 glimpses in training, so the
   full image is an unusual input for it.)

</details>

## D. What does it cost?

A glimpse has 1/16 of the frame's pixels, so k glimpses should cost about k/16 of one pass over the full
image, plus a small LSTM step each. We count the operations (FLOPs) of the encoder on one glimpse and on the
full 512 x 512 frame.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


def gflops(module, x):
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        module(x)
    return fc.get_total_flops() / 1e9


g_glimpse = gflops(seqjepa.encoder, torch.randn(1, 3, FOVEA, FOVEA))
g_full = gflops(seqjepa.encoder, torch.randn(1, 3, FRAME, FRAME))
g_lstm = 2 * 4 * 1408 * (1408 + 1408) / 1e9       # 4 gates, 2 FLOPs per multiply-add (the counter skips the LSTM)
print(f'encoder: one glimpse {g_glimpse:.2f} GFLOPs, full image {g_full:.2f} GFLOPs; LSTM step {g_lstm:.3f} GFLOPs')

### Coding Exercise D.1: the cost of k glimpses

*Exercise objective:* compare the cost of k glimpses with one pass over the full image.

Each glimpse costs one pass of the encoder over a 128 x 128 crop (`g_glimpse`) plus one LSTM step (`g_lstm`).
Complete `glimpse_cost(k)`, the GFLOPs (billions of operations) needed to look at an image with k glimpses. Then:
how many glimpses cost as much as one pass over the full image (`g_full`)?

In [ ]:
def glimpse_cost(k):
    """GFLOPs needed to look at an image with k glimpses."""
    #################################################
    ## TODO: fill in the ... and remove the next line
    raise NotImplementedError('Student exercise: glimpse_cost')
    #################################################
    return ...


# Uncomment to test
# for k in (4, 8, 12):
#     print(f'{k:2d} glimpses: {glimpse_cost(k):.2f} GFLOPs, {glimpse_cost(k) / g_full:.0%} of the full image')

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def glimpse_cost(k):
    return k * (g_glimpse + g_lstm)
```

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('glimpse_cost')
for k in (4, 8, 12):
    print(f'{k:2d} glimpses: {glimpse_cost(k):.2f} GFLOPs, {glimpse_cost(k) / g_full:.0%} of the full image')
print(f'One pass over the full image costs as much as {g_full / glimpse_cost(1):.0f} glimpses.')

## E. Learning where to look

So far the glimpses were placed by fixed rules. Now we want an agent to choose them to solve a given task with reinforcement learning: **find a cued
object**. Each trial is an image and a target category from the 18 categories of COCO-Search18 (a human
visual search dataset). The cue is a picture of *another* object of that category, cut from a different image.

- The eye starts at the centre of the image. The agent then makes up to 8 fixations, each on one of the
  10 x 10 cells; cells that are only black bar are not allowed.
- Reward: +10 when a fixation lands on the target (its cell overlaps the target's outline), which ends the
  trial; -1 for every fixation that misses.

The agent is seq-JEPA (pretraining with uniform action policy so that the performance is not confounded by using saliency maps during pre-training), **frozen**, plus a policy network. After each glimpse, the policy reads the LSTM
state and the cue's representation (from the frozen encoder) and outputs a probability for each of the 100 cells which is the action policy.

Here are three trials. The target is outlined; the cue is on the right of each image.

In [ ]:
SEARCH_CATS = META['search_categories']
CUES_VAL = np.load(os.path.join(DATA_DIR, 'search', 'cues_val.npy'))         # [18, 32, 1280]
CUE_CROPS = np.load(os.path.join(DATA_DIR, 'search', 'cue_crops_val.npy'))   # [18, 32, 3, 128, 128]
N_CUES = (np.abs(CUES_VAL).sum(-1) > 0).sum(1)
CUE_SLOT = np.random.default_rng(1).integers(0, 1 << 30, len(TRIALS)) % N_CUES[[SEARCH_CATS.index(t['category']) for t in TRIALS]]


def trial_axes(n):
    """n (image, cue) axes pairs side by side."""
    fig, axes = plt.subplots(1, 2 * n, figsize=(4.9 * n, 4.2), gridspec_kw={'width_ratios': [4, 1.1] * n})
    for a in axes[1::2]:
        a.axis('off')
    return list(zip(axes[::2], axes[1::2]))


def draw_trial(axs, t, fix_cells=None, title=None):
    ax, cue_ax = axs
    tr = TRIALS[t]
    rec = IMAGES[tr['image']]
    show_image(ax, rec, title=title if title is not None else f"find: {tr['category']}")
    r0, c0, r1, c1 = tr['target_box']
    ax.add_patch(Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, ec='#1baf7a', lw=2.2))
    if fix_cells is not None:
        pts = np.vstack(([[FRAME / 2, FRAME / 2]], CELL_CENTRES[fix_cells]))
        ax.plot(pts[:, 1], pts[:, 0], '-', color='w', lw=1.4, alpha=0.9)
        for i, (r, c) in enumerate(pts):
            hit = i > 0 and fix_cells[i - 1] in tr['target_cells']
            ax.text(c, r, str(i), color='w', fontsize=8, ha='center', va='center', fontweight='bold',
                    bbox=dict(boxstyle='circle,pad=0.25', fc='#1baf7a' if hit else '#2a78d6', ec='w', lw=0.8))
    cue = CUE_CROPS[SEARCH_CATS.index(tr['category']), CUE_SLOT[t]]
    cue_ax.imshow(cue.transpose(1, 2, 0)); cue_ax.set_title('cue', fontsize=9, color=INK2)
    cue_ax.set_anchor('N')


SHOW = (0, 21, 11)                          # trials to draw; try others (0 to len(TRIALS) - 1)
for axs, t in zip(trial_axes(3), SHOW):
    draw_trial(axs, t)
plt.tight_layout(); plt.show()

### Coding Exercise E.1: the reward

*Exercise objective:* write the reward that the agent learns from.

The environment below runs many trials at once, so the reward is computed for a whole batch of trials. After
each fixation, we know for every trial:

- `hit`: `True` if this fixation landed on the target;
- `alive`: `True` if the trial was still running before this fixation. A trial ends when the target is found,
  or after 8 fixations.

The reward is +10 for a hit and -1 for a miss. Trials that had already ended get 0. Complete `search_reward`.

In [ ]:
def search_reward(hit, alive, hit_reward=10.0, miss_reward=-1.0):
    """Reward of one fixation for a batch of trials. hit, alive: [B] True/False. Returns [B] floats."""
    #################################################
    ## TODO: fill in the ... and remove the next line
    raise NotImplementedError('Student exercise: search_reward')
    #################################################
    return ...    # Hint: torch.where(cond, a, b) takes a where cond is True and b elsewhere


# Uncomment to test: should print tensor([10., -1.,  0.])
# print(search_reward(torch.tensor([True, False, False]), torch.tensor([True, True, False])))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def search_reward(hit, alive, hit_reward=10.0, miss_reward=-1.0):
    return torch.where(alive, torch.where(hit, hit_reward, miss_reward), 0.0)
```

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('search_reward')
print(search_reward(torch.tensor([True, False, False]), torch.tensor([True, True, False])))

**The environment.** `Search` runs a batch of trials in parallel. It keeps the LSTM state of each trial,
feeds it the representation of each new glimpse together with the saccade (the action),
and gives the rewards.

In [ ]:
def target_mask(trials):
    m = torch.zeros(len(trials), GRID * GRID, dtype=torch.bool)
    for i, t in enumerate(trials):
        m[i, t['target_cells']] = True
    return m


class Search:
    """Cued search on precomputed glimpse representations emb: [N, 101, 1280] (100 cells, then the image centre)."""
    K, HIT, MISS = 8, 10.0, -1.0

    def __init__(self, model, emb, trials, rows, content, agent='lstm'):
        self.model, self.emb, self.agent = model, emb, agent
        self.rows = torch.as_tensor(rows)                       # row of each trial in emb
        self.target, self.content = target_mask(trials), torch.as_tensor(content)

    def reset(self, idx, cue):
        self.idx, self.cue, B = idx, cue, len(idx)
        self.fix = torch.full((B, 2), FRAME / 2)
        self.alive = torch.ones(B, dtype=torch.bool)
        self.visited = torch.zeros(B, GRID * GRID, dtype=torch.bool)
        self.hit_at = torch.zeros(B, dtype=torch.long)
        self.k = 0
        self._see(torch.from_numpy(self.emb[self.rows[idx], -1].astype(np.float32)), torch.zeros(B, 2))
        return self.obs()

    def _see(self, z, sacc):
        if self.agent == 'lstm':                                 # one LSTM step, with the efference copy
            inp = torch.cat((z, self.model.action(sacc / FRAME)), -1)[:, None]
            out, self.state = self.model.seq_model(inp, None if self.k == 0 else self.state)
            self.h = out[:, 0]
        else:                                                    # no memory: the mean of the glimpse representations
            self.zsum, self.n = (z, 1) if self.k == 0 else (self.zsum + z, self.n + 1)
            self.h = self.zsum / self.n

    def obs(self):
        return torch.cat((self.h, self.cue), -1)

    def mask(self):
        return self.content[self.idx]

    def step(self, cell):
        b = torch.arange(len(cell))
        hit = self.target[self.idx, cell] & self.alive
        reward = search_reward(hit, self.alive, self.HIT, self.MISS)
        self.k += 1
        self.hit_at = torch.where(hit, torch.full_like(self.hit_at, self.k), self.hit_at)
        self.visited[b, cell] = True
        self.alive = self.alive & ~hit & (self.k < self.K)
        nxt = torch.as_tensor(CELL_CENTRES[cell.numpy()], dtype=torch.float32)
        self._see(torch.from_numpy(self.emb[self.rows[self.idx].numpy(), cell.numpy()].astype(np.float32)), nxt - self.fix)
        self.fix = nxt
        return self.obs(), reward


class ActorCritic(nn.Module):
    """The policy (pi: a score per cell) and a value estimate (v), as in our runs."""

    def __init__(self, d, hidden=512):
        super().__init__()
        mlp = lambda out: nn.Sequential(nn.LayerNorm(d), nn.Linear(d, hidden), nn.Tanh(), nn.Linear(hidden, hidden),
                                        nn.Tanh(), nn.Linear(hidden, out))
        self.pi, self.v = mlp(GRID * GRID), mlp(1)

    def dist(self, obs, mask):
        return torch.distributions.Categorical(logits=self.pi(obs).masked_fill(~mask, float('-inf')))

### Watch it learn

We train a policy network with PPO on 354 training trials from other images. Each
update plays 256 trials, then improves the policy towards the fixations that led to more reward than expected.
The full runs used 15,000 trials and 4,000 updates of 1,024 trials each; here you see the first 60 updates. Watch the average return and the share of trials in which the target was found.

In [ ]:
TRAIN = json.load(open(os.path.join(DATA_DIR, 'search', 'train.json')))
EMB_TRAIN = np.load(os.path.join(DATA_DIR, 'search', 'codes_train.npy'))
CUES_TRAIN = torch.from_numpy(np.load(os.path.join(DATA_DIR, 'search', 'cues_train.npy')).astype(np.float32))
CUE_IMG_TRAIN = np.load(os.path.join(DATA_DIR, 'search', 'cue_img_train.npy'))
train_content = torch.zeros(len(TRAIN), GRID * GRID, dtype=torch.bool)
for i, t in enumerate(TRAIN):
    train_content[i, t['content']] = True


def train_cues(idx, g):
    """A random exemplar of each trial's category, never from the trial's own image."""
    out = []
    for i in idx.tolist():
        c = SEARCH_CATS.index(TRAIN[i]['category'])
        ok = np.flatnonzero((CUE_IMG_TRAIN[c] >= 0) & (CUE_IMG_TRAIN[c] != TRAIN[i]['coco_id']))
        out.append(CUES_TRAIN[c, g.choice(ok)])
    return torch.stack(out)


def ppo(env, ac, updates=60, batch=256, lr=3e-4, epochs=4, gamma=0.99, lam=0.95, clip=0.2, seed=SEED):
    opt, g = torch.optim.Adam(ac.parameters(), lr=lr), np.random.default_rng(seed)
    returns, found = [], []
    for u in range(1, updates + 1):
        idx = torch.as_tensor(g.choice(len(env.rows), batch))
        obs = env.reset(idx, train_cues(idx, g))
        O, A, LP, V, R, AL, M = [], [], [], [], [], [], []
        with torch.no_grad():
            for _ in range(env.K):                               # play the trials
                d = ac.dist(obs, env.mask())
                a = d.sample()
                O.append(obs); A.append(a); LP.append(d.log_prob(a)); V.append(ac.v(obs)[:, 0])
                AL.append(env.alive.clone()); M.append(env.mask())
                obs, r = env.step(a)
                R.append(r)
        R, V, AL = torch.stack(R), torch.stack(V), torch.stack(AL)
        nonterm = torch.cat((AL[1:], torch.zeros_like(AL[:1]))).float()
        adv, last = torch.zeros_like(R), torch.zeros(batch)
        for t in reversed(range(env.K)):                         # advantages (GAE)
            nv = V[t + 1] if t + 1 < env.K else torch.zeros(batch)
            last = R[t] + gamma * nv * nonterm[t] - V[t] + gamma * lam * nonterm[t] * last
            adv[t] = last
        keep = AL.flatten()
        flat = lambda x: x.flatten(0, 1)[keep]
        o, a, lp, ad, ret, m = flat(torch.stack(O)), flat(torch.stack(A)), flat(torch.stack(LP)), flat(adv), flat(adv + V), flat(torch.stack(M))
        ad = (ad - ad.mean()) / (ad.std() + 1e-8)
        for _ in range(epochs):                                  # the PPO update
            d = ac.dist(o, m)
            ratio = (d.log_prob(a) - lp).exp()
            loss = (-torch.min(ratio * ad, ratio.clamp(1 - clip, 1 + clip) * ad).mean()
                    + 0.5 * (ac.v(o)[:, 0] - ret).pow(2).mean() - 0.01 * d.entropy().mean())
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(ac.parameters(), 0.5); opt.step()
        returns.append((R * AL).sum(0).mean().item()); found.append((env.hit_at > 0).float().mean().item())
        if u % 10 == 0:
            print(f'updates {u - 9:2d}-{u:2d}: average return {np.mean(returns[-10:]):5.2f}, '
                  f'target found in {np.mean(found[-10:]):.0%} of trials')
    return ac


train_env = Search(seqjepa, EMB_TRAIN, TRAIN, [t['row'] for t in TRAIN], train_content)
t0 = time.time()
_ = ppo(train_env, ActorCritic(1408 + 1280))
print(f'({time.time() - t0:.0f} s)')

### The converged agents

Now we load two policies trained to the end, and test them on 347 test trials.

- **seq-JEPA (no LSTM, avg glimpses)**: the same encoder and the same kind of policy, but its memory is just
  the average of the glimpse representations so far: no LSTM and no action;
- **DeepGaze + IoR**: no learning and no cue; it samples cells from the saliency map of the *whole* image,
  with inhibition of return (the rule of Section A);
- **random**: random cells, never the same cell twice.

The measure is the share of trials in which the target has been fixated by fixation k (target fixation
probability, TFP@k). The agents act greedily: at each step they take their most likely cell.

### Coding Exercise E.2: target fixation probability

*Exercise objective:* compute TFP@k from the results of a search.

After a search, we know for each trial at which fixation the target was found: `hit_at` is 1 to 8, or 0 if
the target was never found. TFP@k is the share of trials in which the target was found at fixation k *or
earlier*. Complete `tfp`, which returns TFP@1, TFP@2, ..., TFP@8.

In [ ]:
def tfp(hit_at, K=8):
    """hit_at: [N] fixation at which each trial found its target (1 ... K), 0 if never. Returns [TFP@1, ..., TFP@K]."""
    #################################################
    ## TODO: fill in the ... and remove the next line
    raise NotImplementedError('Student exercise: tfp')
    #################################################
    return np.array([... for k in range(1, K + 1)])   # Hint: found by fixation k means 0 < hit_at <= k


# Uncomment to test: should print [0.25 0.5  0.5  0.5  0.5  0.5  0.5  0.75]
# print(tfp(np.array([1, 2, 0, 8])))

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def tfp(hit_at, K=8):
    return np.array([((hit_at > 0) & (hit_at <= k)).mean() for k in range(1, K + 1)])
```

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('tfp')
print(tfp(np.array([1, 2, 0, 8])))

In [ ]:
def load_policy(name):
    ck = torch.load(os.path.join(DATA_DIR, 'search', f'policy_{name}.pt'), map_location='cpu')
    ac = ActorCritic(ck['obs_dim'], ck['hidden'])
    ac.pi.load_state_dict({k: v.float() for k, v in ck['state_dict'].items()})
    return ac.eval()


val_content = torch.stack([torch.as_tensor(content_cells(IMAGES[t['image']])) for t in TRIALS])
VAL_ROWS = [t['image'] for t in TRIALS]
VAL_CUES = torch.from_numpy(np.stack([CUES_VAL[SEARCH_CATS.index(t['category']), CUE_SLOT[i]] for i, t in enumerate(TRIALS)]).astype(np.float32))


@torch.no_grad()
def run_search(env, choose, cues=VAL_CUES):
    idx = torch.arange(len(TRIALS))
    obs, cells = env.reset(idx, cues), []
    for _ in range(env.K):
        alive = env.alive.clone()
        c = choose(env, obs)
        obs, _ = env.step(c)
        cells.append(torch.where(alive, c, torch.full_like(c, -1)))
    return env.hit_at.numpy(), torch.stack(cells, 1).numpy()


def greedy(ac):
    return lambda env, obs: ac.dist(obs, env.mask()).probs.argmax(-1)


def random_cells(gen):
    return lambda env, obs: torch.multinomial((env.mask() & ~env.visited).float(), 1, generator=gen)[:, 0]


POOL = np.eye(GRID)[np.arange(SAL_GRID) * GRID // SAL_GRID]               # [128, 10]: saliency row -> cell row


def deepgaze_ior(gen):
    """The sampler of Section A on the action cells: a cell in proportion to its remaining saliency, and IoR on
    the 128 x 128 window around every fixation, the start one included."""
    def inhibit(maps, fix):
        for m, (r, c) in zip(maps, fix):
            r, c, h = int(r), int(c), FOVEA // 2
            m[max(0, (r - h) // CELL):(r + h) // CELL + 1, max(0, (c - h) // CELL):(c + h) // CELL + 1] = 0

    def choose(env, obs):
        if env.k == 0:
            env.sal = np.stack([SALIENCY[IMAGES[r]['sal_row']].astype(np.float64) * content_mask(IMAGES[r]) for r in VAL_ROWS])
            inhibit(env.sal, env.fix.numpy())
        mass = torch.from_numpy(np.einsum('bij,ik,jl->bkl', env.sal, POOL, POOL).reshape(len(env.sal), -1))
        c = torch.multinomial((mass + 1e-12) * env.mask(), 1, generator=gen)[:, 0]
        inhibit(env.sal, CELL_CENTRES[c.numpy()])
        return c
    return choose


def val_env(agent='lstm', model=seqjepa):
    return Search(model, EMB_VAL, TRIALS, VAL_ROWS, val_content, agent)


POLICY = {'lstm': load_policy('lstm'), 'avg': load_policy('avg')}
RESULTS = {'seq-JEPA': run_search(val_env('lstm'), greedy(POLICY['lstm'])),
           'seq-JEPA (no LSTM, avg glimpses)': run_search(val_env('avg'), greedy(POLICY['avg'])),
           'DeepGaze + IoR': run_search(val_env(), deepgaze_ior(torch.Generator().manual_seed(0))),
           'random': run_search(val_env(), random_cells(torch.Generator().manual_seed(0)))}
print(f"{'':34s} TFP@1  TFP@4  TFP@8")
for name, (hit_at, _) in RESULTS.items():
    v = tfp(hit_at)
    print(f'{name:34s} {v[0]:.2f}   {v[3]:.2f}   {v[7]:.2f}')

Some of the agent's searches. Fixation 0 is the start; green means it landed on the target.

In [ ]:
_, cells = RESULTS['seq-JEPA']
for axs, t in zip(trial_axes(3), SHOW):
    draw_trial(axs, t, cells[t][cells[t] >= 0])
plt.tight_layout(); plt.show()

**Our full results**, on the 612 target-present test trials of COCO-Search18 (three training seeds per agent;
the band is ± sd over seeds). DeepGaze + IoR does much better there than on the trials above (0.77 against 0.64
by fixation 8), probably because COCO-Search18 leaves out images with people and animals, which draw saliency
away from the target. **seq-JEPA (no action)** is the model trained
without saccades, with its own LSTM as the agent's memory; **SimCLR (avg glimpses)** is a standard
self-supervised encoder trained on whole images, used like the averaging agent. **Humans** are the COCO-Search18 participants on the same
trials, scored with the same rule (unlike the agents, they also see the rest of the display in their periphery).

In [ ]:
SEARCH = json.load(open(os.path.join(DATA_DIR, 'results', 'search_tfp.json')))['tfp']
CURVES = [('humans', 'humans', INK, 'P'),                                    # in the order of TFP@8
          ('lstm_sj_uniform', 'seq-JEPA', TRAIN_COLORS['uniform'], 'o'),
          ('deepgaze_ior', 'DeepGaze + IoR', TRAIN_COLORS['saliency + IoR'], 'v'),
          ('lstm_sj_noact', 'seq-JEPA (no action)', TRAIN_COLORS['no action'], 's'),
          ('mean_sj_uniform', 'seq-JEPA (no LSTM, avg glimpses)', '#7d93f0', 'D'),
          ('mean_simclr', 'SimCLR (avg glimpses)', '#8a6a00', '^'),
          ('random', 'random', INK3, 'X')]
ks = np.arange(1, 9)
fig, ax = plt.subplots(figsize=(9.5, 5))
ax.grid(axis='y', color=GRIDC, lw=0.8)
for key, label, color, marker in CURVES:
    v = np.array(SEARCH[key])
    m = v.mean(0)
    if len(v) > 1:                                         # several training seeds: mean +- sd
        s = v.std(0, ddof=1)
        ax.fill_between(ks, m - s, m + s, color=color, alpha=0.15, lw=0)
    ax.plot(ks, m, '-', lw=2, color=color, marker=marker, ms=6.5, mec='w', mew=0.8, label=label)
ax.set_xlabel('fixation k'); ax.set_ylabel('target found by fixation k'); ax.set_ylim(0, 1); ax.set_xticks(ks)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)
plt.tight_layout(); plt.show()

### Coding Exercise E.3: take away the efference copy

*Exercise objective:* find out what the agent uses the efference copy for.

The agent's LSTM receives each glimpse together with the saccade that brought the eye there (the efference
copy). What happens if, at test time, we keep everything else the same but give the LSTM a zero saccade action? (the policy is not retrained)

First, predict: will the agent find fewer targets? Will its fixations change in some other way?

Then complete `search_without_efference_copy`. The model has a switch for this, `seqjepa.use_action` (see
`SeqJEPA.action` in Section B). `compare_efference_copy()` prints TFP@8 and the *revisit rate*: the share of
fixations that land on a cell the agent had already fixated in the same trial.

In [ ]:
def revisit_rate(cells):
    """Share of fixations on a cell already fixated in the same trial. cells: [N, 8], -1 once the trial is over."""
    n = r = 0
    for row in cells:
        row = row[row >= 0]
        n, r = n + len(row), r + len(row) - len(set(row.tolist()))
    return r / n


def search_without_efference_copy():
    """The converged seq-JEPA agent on the pack's trials, with a zero saccade going into its LSTM."""
    #################################################
    ## TODO: fill in the ... and remove the next line
    raise NotImplementedError('Student exercise: search_without_efference_copy')
    #################################################
    seqjepa.use_action = ...                                       # switch off the saccade input
    hit_at, cells = run_search(val_env('lstm'), greedy(POLICY['lstm']))
    seqjepa.use_action = True                                      # and back on, for the rest of the notebook
    return hit_at, cells


def compare_efference_copy():
    for name, (hit_at, cells) in (('with the efference copy', RESULTS['seq-JEPA']),
                                  ('without it', search_without_efference_copy())):
        print(f'{name:24s} TFP@8 {tfp(hit_at)[7]:.2f}, revisits {revisit_rate(cells):.0%}')


# Uncomment to test
# compare_efference_copy()

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def search_without_efference_copy():
    seqjepa.use_action = False
    hit_at, cells = run_search(val_env('lstm'), greedy(POLICY['lstm']))
    seqjepa.use_action = True
    return hit_at, cells
```

Without the saccade input, the agent finds fewer targets, and the revisits show why: it no longer knows where
it has already looked, so it keeps going back to the same cells. The policy relies on the LSTM's record of
where the eye has been, and the LSTM builds that record from the efference copy (Exercise C.1).

</details>

In [ ]:
# Run this cell to use our solution (stored in the data pack) so the rest of the notebook runs; skip it to keep yours
load_solution('search_without_efference_copy')
compare_efference_copy()

## What to take away from this hands-on:

- Glimpses are cheap: eight of them cost 57% of one pass over the full image.
- A model can learn a useful scene memory from glimpses alone, by predicting what it will see after each
  saccade. Knowing its own saccades (the efference copy) is what lets it keep track of *where* things are.
- That memory is a good basis for learning where to look: a small policy on top of it searches better than
  the same encoder without the memory.
- What a model sees depends on how it looks. The same model reads out more about an image when its glimpses
  are placed well.